# Phase 2 & 3: Data Cleaning & Feature Creation
* **Project:** Seasonal Analysis of Academic AI Evasion Tools
* **Data Source:** Google Trends (Past 5 years, US)

### Objective
Clean raw search volume data, fix Google Trends formatting issues, convert date strings into date objects, and create new columns for summer months and total search intensity.

## Import Libraries and Data Ingestion
Loading core data manipulation libraries and importing the raw Google Trends CSV. We use 'skiprows=1' to remove Google's default export metadata so the dataset loads cleanly.

In [5]:
import pandas as pd
import numpy as np

# Load the raw Google Trends CSV file
df = pd.read_csv("academic_ai_evasion_trends.csv")

# Standardize column names to remove spaces and special characters for easy referencing
df.columns = ["Month", "Quillbot", "Bypass_AI", "Turnitin_AI", "AI_Humanizer"]

# Inspect the first 5 rows of the raw dataset
df.head()

,Month,Quillbot,Bypass_AI,Turnitin_AI,AI_Humanizer
0,2021-08-01,8,0,10,0
1,2021-09-01,17,0,16,0
2,2021-10-01,20,0,14,0
3,2021-11-01,22,0,13,0
4,2021-12-01,18,0,13,0


## Data Type Cleaning & Formatting
Renaming messy column names into clean labels, converting the 'Month' column into Python date objects, and replacing Google's low-volume search text (<1) with the number 0.

In [6]:
# Convert 'Month' string column into proper datetime format for time-series analysis
df["Month"] = pd.to_datetime(df["Month"])

# Handle low search volume markers: replace string '<1' values with numerical 0
df = df.replace("<1", 0)

# Ensure all AI tool columns are explicitly converted to numeric data types
tools = ["Quillbot", "Turnitin_AI", "Bypass_AI", "AI_Humanizer"]
for tool in tools:
    df[tool] = pd.to_numeric(df[tool])

# Inspect structural summary to confirm clean data types (datetime64 and int64/float64)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Month         61 non-null     datetime64[ns]
 1   Quillbot      61 non-null     int64         
 2   Bypass_AI     61 non-null     int64         
 3   Turnitin_AI   61 non-null     int64         
 4   AI_Humanizer  61 non-null     int64         
dtypes: datetime64[ns](1), int64(4)
memory usage: 2.5 KB


## Feature Engineering
Adding useful analytical columns: a summer flag (is_summer) to tag non-school months, and a combined (evasion_intensity) score that adds up search interest across all bypass tools.

In [7]:
# Create a boolean flag for summer months (June = 6, July = 7, August = 8)
df["is_summer"] = df["Month"].dt.month.isin([6, 7, 8])

# Calculate total evasion intensity across all non-Quillbot AI humanizers/bypassers
df["evasion_intensity"] = df["Turnitin_AI"] + df["Bypass_AI"] + df["AI_Humanizer"]

# View updated dataframe with new feature columns
df.head()

,Month,Quillbot,Bypass_AI,Turnitin_AI,AI_Humanizer,is_summer,evasion_intensity
0,2021-08-01,8,0,10,0,True,10
1,2021-09-01,17,0,16,0,False,16
2,2021-10-01,20,0,14,0,False,14
3,2021-11-01,22,0,13,0,False,13
4,2021-12-01,18,0,13,0,False,13


## Verification & CSV Export
Checking summary statistics to confirm data integrity (no missing values, correct numbers), then saving the cleaned data as "cleaned_academic_ai_evasion_trends.csv".

In [8]:
# Summary statistics to verify clean data and metrics
print(df.describe())

# Save cleaned dataset to CSV for the next stage (Visualization)
df.to_csv("cleaned_academic_ai_evasion_trends.csv", index=False)
print("Data processing complete. Cleaned file exported successfully!")

                               Month    Quillbot  Bypass_AI  Turnitin_AI  \
count                             61   61.000000  61.000000    61.000000   
mean   2024-01-31 02:45:14.754098432   49.311475   2.721311    25.688525   
min              2021-08-01 00:00:00    8.000000   0.000000     6.000000   
25%              2022-11-01 00:00:00   28.000000   0.000000    16.000000   
50%              2024-02-01 00:00:00   47.000000   2.000000    27.000000   
75%              2025-05-01 00:00:00   69.000000   3.000000    35.000000   
max              2026-08-01 00:00:00  100.000000  12.000000    44.000000   
std                              NaN   24.304692   3.055766    10.490537   

       AI_Humanizer  evasion_intensity  
count     61.000000          61.000000  
mean      14.245902          42.655738  
min        0.000000           6.000000  
25%        0.000000          20.000000  
50%        3.000000          35.000000  
75%       20.000000          62.000000  
max       62.000000         